In [17]:
import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Veriyi yükleme
df = pd.read_csv("data/lip_coordinates.csv")
df.set_index("time", inplace=True)

# Eğitim ve test kümelerini belirleme (%80 eğitim, %20 test)
train_size = int(len(df) * 0.8)
train, test = df.iloc[:train_size], df.iloc[train_size:]

# Performans metriklerini hesaplama fonksiyonu
def calculate_metrics(true_values, predicted_values):
    mae = mean_absolute_error(true_values, predicted_values)
    mse = mean_squared_error(true_values, predicted_values)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((true_values - predicted_values) / true_values)) * 100
    return mae, mse, rmse, mape

# VAR Modeli ile tahmin yapma
var_model = VAR(train)
var_result = var_model.fit(maxlags=5)  # Otomatik gecikme seçimi

# Test seti için tahmin yapma
predictions_var = var_result.forecast(train.values[-5:], steps=len(test))
predictions_var_df = pd.DataFrame(predictions_var, index=test.index, columns=df.columns)

# Gerçek ve tahmin edilen değerler için metrikleri hesaplama
mae_var, mse_var, rmse_var, mape_var = calculate_metrics(test.values, predictions_var)

# Sonuçları ekrana yazdırma
print(f"VAR Modeli Sonuçları:\nMAE: {mae_var:.4f}, MSE: {mse_var:.4f}, RMSE: {rmse_var:.4f}, MAPE: {mape_var:.2f}%")

# Gerçek ve tahmin edilen değerleri ekrana yazdırma
print("\nGerçek Değerler vs Tahmin Edilen Değerler:")
comparison_df = pd.concat([test.reset_index(), predictions_var_df.reset_index()], axis=1, keys=["Gerçek Değerler", "Tahmin Edilen Değerler"])
print(comparison_df.head(10))  # İlk 10 satırı gösterme

C:\Users\Gozde\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


VAR Modeli Sonuçları:
MAE: 13.2608, MSE: 272.0270, RMSE: 16.4932, MAPE: 3.21%

Gerçek Değerler vs Tahmin Edilen Değerler:
  Gerçek Değerler                                               ...  \
             time  0_x 13_x 14_x 17_x 37_x 39_x 40_x 61_x 78_x  ...   
0           35.16  631  632  632  633  621  611  604  597  602  ...   
1           35.20  630  630  631  632  620  610  604  597  602  ...   
2           35.24  628  629  629  630  618  608  602  596  601  ...   
3           35.28  627  628  628  629  618  608  602  596  601  ...   
4           35.32  627  628  629  630  618  608  602  596  602  ...   
5           35.36  627  627  628  629  617  607  601  595  601  ...   
6           35.40  627  627  628  629  616  607  601  595  601  ...   
7           35.44  626  627  628  629  616  606  600  594  601  ...   
8           35.48  626  627  628  629  616  607  602  597  603  ...   
9           35.52  625  626  627  628  615  606  601  597  603  ...   

  Tahmin Edilen Değerler 